# Ft Huachucah Canopy Dataset
This dataset comes from canopy measurements taken in 2024 in the Ft Huachucah sentinel landscape.

In [1]:
# import calc_biomass package
import sys
sys.path.append('../')
from calc_biomass import manage_data

import pandas as pd

In [2]:
data = manage_data.LoadData()
data.load_data(data.data_path)

Adjust column names to match a standard format: DBH, spp. Height is converted automatically.

In [3]:
data.df.rename(columns={'Final DBH/DRC':'dbh', 'spp':'Species','Species Code':'spp'}, inplace=True)

## Calculating Equivalent DRC

Using standard protocol for Diameter at Root Collar (DRC), when multiple stems joined to a single root collar below the soil, individual stems were measured for the same tree. These multiple basal measurements must be combined into a single measurement using the equation:

equivalent diameter = $\sqrt{\sum_{i=1}^{n}}RCD_{i}^{2}$

This equation is provided in both of:

    Chojnakcy, D.C. 1992 . Estimating volume and biomass for dryland oak speices. In: Ffolliott P.F., Gottfried, G.J., Bennett, D.A., Hernandez, C.V.-M., Ortega-Rubio, A., and R.H. Hamre, technical coordinators. Ecology and management of oak and associated woodlands: perspectives in the southwestern United States and Northern Mexico. Proceedings April 7-30, 1992; Sierra Vista, AZ. USDA Forest Service, Rocky Mountain Forest and Range Experiment Station General Technical Report RM-218:155-161.

    Grier, C.C., Elliott, K.J., McCullough, D.G. 1992. Biomass distribution and productivity of Pinus edulis-Juniperus monosperma woodlands of north-central Arizona. Forest Ecology and Management, 50:331-350.


### Keeping Other Measurements
Height, for example, is only entered for once for each tree, even when there are multiple stems. It is important to make sure that the height value is retained and not reverted to a NAN. Look at this example:

In [4]:
test = data.df[(data.df.Plot==358)&(data.df['Tree Number']==17)]
test

,Plot,Quadrant,Species,spp,Tree Number,DBH/DRC calculated,DBH/DRC,dbh,Unnamed: 8,DBH/DRC Circumfrence (cm),Height,Base reading,Top Reading,Crown Class,Lowest Canopy Height,Lowest Canopy Height Reading,Tree Condition,Distance (M),Stems
58,358,SE,Silver Leaf Oak,QUHY,17.0,1.127870,DRC,1.12787,0.0,9.0,2.14,-39.0,1.0,S,1.0165,-20.0,1.0,5.35,NaN
99,358,SE,Silver Leaf Oak,QUHY,17.0,1.127870,DRC,1.12787,0.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100,358,SE,Silver Leaf Oak,QUHY,17.0,1.879783,DRC,1.879783,0.0,15.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Below, we confirm that the max is the best way to preserve this information, including string information like species and quadrant. Only one NAN value is propagated.

In [5]:
other_col = ~test.columns.isin(['dbh'])
test.loc[:, other_col].max()

Plot                                        358
Quadrant                                     SE
Species                         Silver Leaf Oak
spp                                        QUHY
Tree Number                                17.0
DBH/DRC calculated                     1.879783
DBH/DRC                                     DRC
Unnamed: 8                                  0.0
DBH/DRC Circumfrence (cm)                  15.0
Height                               2.14 meter
Base reading                              -39.0
Top Reading                                 1.0
Crown Class                                   S
Lowest Canopy Height               1.0165 meter
Lowest Canopy Height Reading              -20.0
Tree Condition                              1.0
Distance (M)                               5.35
Stems                                       NaN
dtype: object

### Summmarizing by tree

In [6]:
def calc_edrc(df):
    # df is one tree id from a single plot
    # for trees with only one tree id, the sqrt(d2) == d
    dbh = (sum(df['dbh']**2))**0.5
    
    other_val = df.max()

    newdf = pd.DataFrame()
    newdf.loc[1,'DBH'] = dbh
    newdf.loc[1, df.columns] = other_val

    return newdf

In [7]:
# for each tree in each plot, calculate the equivalent edrc
# then flatten the group multi-indeces back into columns
# for trees with only one tree id, the sqrt(d2) == d
data.df = data.df.groupby(['Plot', 'Tree Number']).apply(calc_edrc).reset_index()

## Checking for negative, missing, and 0 values

It is important to check that all the data is within a valid range and that no plots are missing any data. This is especially important for the 3 datapoints used to calculate biomass: species, dbh, and height.

### Species

In [8]:
data.df[data.df.spp.isna()]

,Plot,Tree Number,level_2,DBH,Quadrant,Species,spp,DBH/DRC calculated,DBH/DRC,dbh,...,DBH/DRC Circumfrence (cm),Height,Base reading,Top Reading,Crown Class,Lowest Canopy Height,Lowest Canopy Height Reading,Tree Condition,Distance (M),Stems
298,3301,6.0,1,4.386159848989242 inch,SW,Pinyon,NaN,4.38616,DBH,4.386159848989242 inch,...,35.0,4.1205 meter,-20.0,21.0,I,2.0100000000000002 meter,NaN,3.0,10.05,NaN


This will be filled with a common pinyon ID for *pinus edulis*.

In [9]:
pied = data.df.spp.isna() & (data.df.Species=='Pinyon')
data.df.loc[pied, 'spp'] = 'PIED'

### DBH

In [10]:
data.df[data.df.DBH<=0]

,Plot,Tree Number,level_2,DBH,Quadrant,Species,spp,DBH/DRC calculated,DBH/DRC,dbh,...,DBH/DRC Circumfrence (cm),Height,Base reading,Top Reading,Crown Class,Lowest Canopy Height,Lowest Canopy Height Reading,Tree Condition,Distance (M),Stems


In [11]:
data.df[data.df.DBH.isna()]

,Plot,Tree Number,level_2,DBH,Quadrant,Species,spp,DBH/DRC calculated,DBH/DRC,dbh,...,DBH/DRC Circumfrence (cm),Height,Base reading,Top Reading,Crown Class,Lowest Canopy Height,Lowest Canopy Height Reading,Tree Condition,Distance (M),Stems


### Height

In [12]:
data.df[data.df.Height<=0]

,Plot,Tree Number,level_2,DBH,Quadrant,Species,spp,DBH/DRC calculated,DBH/DRC,dbh,...,DBH/DRC Circumfrence (cm),Height,Base reading,Top Reading,Crown Class,Lowest Canopy Height,Lowest Canopy Height Reading,Tree Condition,Distance (M),Stems
249,3102,1.0,1,16.291450867674328 inch,NW,Velvet Mesquite,PRVE,16.291451,NaN,16.291450867674328 inch,...,130.0,-44.4 meter,296.0,NaN,C,-35.550000000000004 meter,59.0,2.0,15.0,NaN
250,3102,2.0,1,11.77997216585682 inch,NW,Velvet Mesquite,PRVE,11.779972,NaN,11.77997216585682 inch,...,94.0,-24.09 meter,219.0,NaN,I,-19.580000000000002 meter,41.0,2.0,11.0,NaN
251,3102,3.0,1,8.521681992321955 inch,NW,Velvet Mesquite,PRVE,8.521682,NaN,8.521681992321955 inch,...,68.0,-23.31 meter,259.0,NaN,I,-18.900000000000002 meter,49.0,2.0,9.0,NaN
252,3102,4.0,1,11.278696754543764 inch,NW,Velvet Mesquite,PRVE,11.278697,NaN,11.278696754543764 inch,...,90.0,-30.520000000000003 meter,218.0,NaN,I,-27.3 meter,23.0,2.0,14.0,NaN
253,3102,5.0,1,3.7595655848479215 inch,NW,Velvet Mesquite,PRVE,3.759566,NaN,3.7595655848479215 inch,...,30.0,-8.72 meter,218.0,NaN,S,-6.36 meter,59.0,2.0,4.0,NaN
254,3102,6.0,1,5.012754113130562 inch,SW,Velvet Mesquite,PRVE,5.012754,NaN,5.012754113130562 inch,...,40.0,-15.299999999999999 meter,255.0,NaN,C,-12.0 meter,55.0,2.0,6.0,NaN
255,3102,7.0,1,3.99450771837772 inch,SW,Velvet Mesquite,PRVE,3.508928,NaN,3.5089278791913934 inch,...,28.0,-10.649999999999999 meter,213.0,NaN,I,-7.35 meter,66.0,2.0,5.0,NaN
256,3102,8.0,1,13.785073811109045 inch,SW,Velvet Mesquite,PRVE,13.785074,NaN,13.785073811109045 inch,...,110.0,-25.92 meter,216.0,NaN,I,-17.52 meter,70.0,2.0,12.0,NaN
257,3102,9.0,1,11.028059048887236 inch,SE,Velvet Mesquite,PRVE,11.028059,NaN,11.028059048887236 inch,...,88.0,-24.97 meter,227.0,NaN,I,-16.830000000000002 meter,74.0,2.0,11.0,NaN
258,3102,10.0,1,8.271044286665427 inch,SE,Velvet Mesquite,PRVE,8.271044,NaN,8.271044286665427 inch,...,66.0,-23.17 meter,331.0,NaN,D,-17.92 meter,75.0,2.0,7.0,NaN


#### Mesquite 

After consulting with Zito, all of the negative heights associated with mesquite are due to height being calculated off of clinometer measurements. In the case of mesquite, the clinometer base reading is actually measured height in cm. I make the adjutment to the data set below.

In [13]:
prve = (data.df['spp']=='PRVE')&(data.df['Height']<=0)
data.df.loc[prve, 'Height'] = data.df.loc[prve, 'Base reading']/100

#### Alligator juniper

After working with Zito to consult the original plot sheets, the alligator juniper had a mis-entered upper clinometer measurement, I recalculate below.

In [14]:
jude = (data.df['spp']=='JUDE2')&(data.df['Height']<0)
ht_calc = ['dbh', 'Base reading', 'Top Reading', 'Distance (M)']
data.df.loc[jude, ht_calc]

,dbh,Base reading,Top Reading,Distance (M)
368,4.230554788801639 inch,-11.0,-18.0,7.1


In [15]:
tp = -data.df.loc[jude, 'Top Reading']
btm = data.df.loc[jude, 'Base reading']
dist = data.df.loc[jude, 'Distance (M)']

ht = (tp-btm) / 100 * dist

data.df.loc[jude, 'Height'] = ht

#### Remaining
Only 1 tree remains in the dataset without a height. It is not a species of high concern, so I will leave it be.

In [16]:
data.df[data.df.Height<=0]

,Plot,Tree Number,level_2,DBH,Quadrant,Species,spp,DBH/DRC calculated,DBH/DRC,dbh,...,DBH/DRC Circumfrence (cm),Height,Base reading,Top Reading,Crown Class,Lowest Canopy Height,Lowest Canopy Height Reading,Tree Condition,Distance (M),Stems
508,6017,1.0,1,1.1278696754543764 inch,SE,Acacia,SEGR10,1.12787,NaN,1.1278696754543764 inch,...,9.0,0.0 meter,NaN,NaN,D,0.0 meter,NaN,2.0,NaN,20.0


#### NAN

In [17]:
data.df[data.df.Height.isna()]

,Plot,Tree Number,level_2,DBH,Quadrant,Species,spp,DBH/DRC calculated,DBH/DRC,dbh,...,DBH/DRC Circumfrence (cm),Height,Base reading,Top Reading,Crown Class,Lowest Canopy Height,Lowest Canopy Height Reading,Tree Condition,Distance (M),Stems


## Calculating Tree Biomass

In [18]:
wt = manage_data.CalcBiomass(data)
# what equations should be used for these species?
wt.add_equ_column()

In [19]:
wt.calc_biomass('total')

140: RuntimeWarning: invalid value encountered in log10


### OakWoodland_Chojnacky error

This error comes from the `OakWoodland_Chojnacky` class in `allometry.py`:
```python
    def _eq_wght_crwn(self, wght_bd, coefs):
        B0, B1, B2 = coefs
        return 10**(B0 + B1*np.log10(wght_bd) + B2*self.trees['HT'])
```
Let's investigate this error to make sure we are still getting valid weights from this equation for our ouak species.

In [43]:
wt.ldata.df[['Species', 'equ', 'total_wt']]

,Species,equ,total_wt
0,Alligator Juniper,PJ_Grier,138.568103
1,Alligator Juniper,PJ_Grier,477.141235
2,Alligator Juniper,PJ_Grier,117.457209
3,SW White Pine,BCtimber_Standish,0.0
4,Ponderosa Pine,BCtimber_Standish,154.439431
...,...,...,...
504,Creosote,NaN,NaN
505,Creosote,NaN,NaN
506,Creosote,NaN,NaN
507,Velvet Mesquite,NaN,NaN


#### Missing biomass
Let's tackle the missing biomass first.

In [45]:
eq = wt.ldata.df.equ == 'OakWoodland_Chojnacky'
wt.ldata.df.loc[eq, ['Species', 'equ', 'total_wt']]

,Species,equ,total_wt
8,Mexican Blue Oak,OakWoodland_Chojnacky,NaN
14,Mexican Blue Oak,OakWoodland_Chojnacky,NaN
20,AZ White Oak,OakWoodland_Chojnacky,NaN
21,AZ White Oak,OakWoodland_Chojnacky,NaN
23,AZ White Oak,OakWoodland_Chojnacky,NaN
...,...,...,...
420,Mexican Blue Oak,OakWoodland_Chojnacky,NaN
421,Mexican Blue Oak,OakWoodland_Chojnacky,NaN
462,Mexican Blue Oak,OakWoodland_Chojnacky,NaN
466,Net Leaf Oak,OakWoodland_Chojnacky,NaN


In [46]:
eq =  'OakWoodland_Chojnacky'
oak = wt.calc_equ(wt.ldata.df[wt.ldata.df.equ ==eq], eq,'total')
oak

140: RuntimeWarning: invalid value encountered in log10


,weight
8,NaN
14,NaN
20,NaN
21,NaN
23,NaN
...,...
420,NaN
421,NaN
462,NaN
466,NaN


In [47]:
oak.isna().describe()

,weight
count,177
unique,1
top,True
freq,177


All oaks are being assigned an NAN value. We need to dig through the allometry equation.

In [32]:
from calc_biomass import allometry

In [51]:
# first get a clean df with the right units and column names
owc_cls = getattr(allometry, 'OakWoodland_Chojnacky')
df_slice = wt.ldata.df[wt.ldata.df.equ =='OakWoodland_Chojnacky']
df_clean = wt.get_clean_df(df_slice, owc_cls.required_columns, owc_cls.units )

In [52]:
ow = allometry.OakWoodland_Chojnacky(df_clean)

In [53]:
ow.calc_tot_tree_weight()

140: RuntimeWarning: invalid value encountered in log10


8       419.666496
14       90.062109
20       18.066674
21       81.157153
23       34.098197
          ...     
420     100.988665
421     101.329458
462     628.808909
466     176.658916
470    1114.556803
Name: HT, Length: 177, dtype: Float64

That didn't flesh out the warning, but it indicates that :meth:`manage_data.CalcBiomass.calc_equ` is getting valid output...So that's the next check.

This took a lot of work in debugger, but this equation:

```python
    def _eq_wght_crwn(self, wght_bd, coefs):
        B0, B1, B2 = coefs
        return 10**(B0 + B1*np.log10(wght_bd) + B2*self.trees['HT'])
```

transformed the output of allometry into a new `pd.Series` when it used the value from `self.trees` with a new index. So when :meth:`manage_data.CalcBiomass.calc_equ` went to apply units to it in this line:

```python
        wt = pd.DataFrame(data=wt, columns=['weight'], index=data.index)
        wt['weight'] = UnitsMngr(wt, cls_inst.units).assign_unit('weight', cls_inst.units['out']['weight'])
```

It reindexed everything, and since that didn't line up with the index of the slice, everything ended up as an NaN value...

Let's reload and try again.

In [122]:
%load_ext autoreload
%autoreload explicit
%aimport calc_biomass.manage_data
import calc_biomass.manage_data

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [125]:
del wt
del data

In [107]:
del sys.modules['calc_biomass']
del sys.modules['calc_biomass.allometry']
del sys.modules['calc_biomass.manage_data']

from calc_biomass import manage_data

In [126]:
# load data
data = manage_data.LoadData()
data.load_data(data.data_path)

# calculate ERC
data.df.rename(columns={'Final DBH/DRC':'dbh', 'spp':'Species','Species Code':'spp'}, inplace=True)
data.df = data.df.groupby(['Plot', 'Tree Number']).apply(calc_edrc).reset_index()

# clean data
pied = data.df.spp.isna() & (data.df.Species=='Pinyon')
data.df.loc[pied, 'spp'] = 'PIED'

prve = (data.df['spp']=='PRVE')&(data.df['Height']<=0)
data.df.loc[prve, 'Height'] = data.df.loc[prve, 'Base reading']/100

jude = (data.df['spp']=='JUDE2')&(data.df['Height']<0)
ht = (-data.df.loc[jude, 'Top Reading']- data.df.loc[jude, 'Base reading']) / 100 * data.df.loc[jude, 'Distance (M)']
data.df.loc[jude, 'Height'] = ht

wt = manage_data.CalcBiomass(data)
# what equations should be used for these species?
wt.add_equ_column()

Attempts to reload the package have failed. To preserve the state of the data investigated above, a new notebook will be created and so the package can be loaded fresh.